# Appendix calculation: Bloch-periodic one-electron hydrogenic lattice

This notebook is independent of the main paper Colab notebook and the main mrsr Dirichlet solver. It was added only for the periodic/Bloch-periodic reviewer-response calculation. Existing hydrogen and H2+ workflows are not modified.

The purpose is narrow: demonstrate that the same normalized matrix-free Rayleigh-residual relaxation works when the local finite-difference stencil is supplied with periodic or Bloch-periodic boundary links.

The calculation is a single-particle periodic hydrogenic lattice model, not a quantitative many-electron calculation of metallic hydrogen.


# Mathematical setup

The periodic cell contains N^3 unique grid points with h=a/N; the endpoint at a is not stored separately.

At Bloch vector k, only stencil links crossing a cell face are modified. Across the positive alpha face, the wrapped value receives exp(+i k_alpha a); across the negative face it receives exp(-i k_alpha a).

The relaxation remains

R_n = (H_h(k)-rho_n I) psi_n

q_(n+1) = psi_n - tau R_n

psi_(n+1) = q_(n+1) / ||q_(n+1)||

The only formal change relative to the real Dirichlet derivation is the complex inner product P^T Q -> P^dagger Q.


# Periodic hydrogenic potential

For one proton-like center at R_p in a cubic cell of volume Omega=a^3, the periodic Coulomb Green function is represented on the grid by

V_G = -8 pi exp(-i G.R_p) / (Omega |G|^2),  G != 0,

with V_0=0 in Rydberg units. The omitted zero mode fixes the arbitrary mean potential (equivalently, the neutralized periodic Green-function convention). The inverse FFT evaluates the reciprocal-space Fourier series on the real-space grid.

An odd N is used so that the proton at (a/2,a/2,a/2) lies halfway between grid nodes in all three directions.


# 1. Colab setup

Use a GPU runtime. All appendix outputs are written to a separate directory.


In [ ]:
from pathlib import Path
import os, sys

REPO = Path('/content/matrix-free-rayleigh-relaxation')
if not REPO.exists():
    !git clone -q https://github.com/mahdi-sasar/matrix-free-rayleigh-relaxation.git /content/matrix-free-rayleigh-relaxation
else:
    %cd /content/matrix-free-rayleigh-relaxation
    !git pull -q

%cd /content/matrix-free-rayleigh-relaxation
!pip install -q -r requirements-colab.txt

import json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from appendix_periodic.periodic_hydrogen import (
    periodic_coulomb_potential,
    validate_free_electron,
    hermiticity_error,
    solve_ground_state,
    high_symmetry_path,
    run_band_path,
    plot_band_structure,
    plot_tiled_density,
    plot_periodic_potential_cut,
)

print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))


# 2. Run parameters

Quick mode is intended for structural validation. Production mode is intended for the appendix figure.


In [ ]:
PRODUCTION = False

A_BOHR = 5.0
SIGMA = 0.80

if PRODUCTION:
    N = 49
    POINTS_PER_SEGMENT = 13
    TOL = 1e-8
    MAX_ITER = 120_000
    CHECK_EVERY = 20
else:
    N = 33
    POINTS_PER_SEGMENT = 7
    TOL = 1e-7
    MAX_ITER = 50_000
    CHECK_EVERY = 50

H = A_BOHR / N
OUT = Path('/content/periodic_hydrogen_appendix_results')
OUT.mkdir(parents=True, exist_ok=True)

print(f'N={N}, a={A_BOHR:.3f} Bohr, h={H:.6f} Bohr')


# 3. Construct and inspect the periodic Coulomb potential

The mean potential should vanish to roundoff because the G=0 coefficient is set to zero. The inverse FFT should be real to roundoff.


In [ ]:
V_np, V_meta = periodic_coulomb_potential(N, A_BOHR)
print(json.dumps(V_meta, indent=2))

assert abs(V_meta['mean_potential_Ry']) < 1e-10
assert V_meta['max_imaginary_roundoff_Ry'] < 1e-10 * max(1.0, abs(V_meta['potential_min_Ry']))
assert V_meta['nearest_grid_distance_to_proton_Bohr'] > 0.0

plot_periodic_potential_cut(
    V_np, A_BOHR,
    OUT / 'periodic_hydrogen_potential_cut.png',
    show=True,
)


# 4. Exact free-electron Bloch test

For V=0, a sampled plane wave is an exact eigenvector of the second-order stencil. Its discrete energy is

E_h(k) = (4/h^2) sum_alpha sin^2(k_alpha h/2).

This checks the Bloch phase signs, complex inner product, and wrapped stencil independently of the relaxation.


In [ ]:
pi_over_a = np.pi / A_BOHR
free_k = [
    (0.0, 0.0, 0.0),
    (0.37*pi_over_a, 0.0, 0.0),
    (0.41*pi_over_a, 0.23*pi_over_a, 0.0),
    (0.33*pi_over_a, 0.27*pi_over_a, 0.19*pi_over_a),
]
free_test = validate_free_electron(N, A_BOHR, free_k)
display(free_test)

assert free_test['abs_error_Ry'].max() < 1e-10
assert free_test['normalized_residual'].max() < 1e-10
print('PASS: Bloch-wrapped stencil reproduces the exact discrete free-electron dispersion.')


# 5. Hermiticity test

At nonzero k the Hamiltonian is complex, but it must satisfy <A|H B> = <H A|B>.


In [ ]:
k_generic = (0.37*pi_over_a, 0.21*pi_over_a, 0.13*pi_over_a)
herr = hermiticity_error(V_np, A_BOHR, k_generic)
print(f'Relative Hermiticity error = {herr:.3e}')
assert herr < 1e-11
print('PASS: matrix-free Bloch Hamiltonian is Hermitian to numerical precision.')


# 6. Gamma-point periodic state and Rayleigh descent

At Gamma all boundary phases equal one. This is ordinary periodic wrapping: the neighbor beyond one face is taken from the opposite face.


In [ ]:
GAMMA = (0.0, 0.0, 0.0)
gamma = solve_ground_state(
    V_np, A_BOHR, GAMMA,
    sigma=SIGMA,
    tolerance=TOL,
    max_iterations=MAX_ITER,
    check_every=CHECK_EVERY,
    verbose=True,
)

print(f'energy     = {gamma.energy:+.12f} Ry')
print(f'residual   = {gamma.residual:.3e}')
print(f'iterations = {gamma.iterations}')
print(f'converged  = {gamma.converged}')
print(f'tau        = {gamma.tau:.6e}')

assert gamma.converged
dE = np.diff(gamma.energy_history)
max_upward = float(np.max(dE)) if len(dE) else 0.0
print(f'max sampled upward energy change = {max_upward:.3e} Ry')
assert max_upward < 5e-11


# 7. Lowest Bloch band along Gamma-X-M-R-Gamma

Only the lowest eigenstate is computed at each k. Thus every individual calculation remains a ground-state relaxation problem; no multi-eigenvector solver is introduced.


In [ ]:
path = high_symmetry_path(A_BOHR, POINTS_PER_SEGMENT)

band_df, band_results = run_band_path(
    V_np,
    A_BOHR,
    path,
    psi0=gamma.psi,
    sigma=SIGMA,
    tolerance=TOL,
    max_iterations=MAX_ITER,
    check_every=CHECK_EVERY,
    verbose=False,
)

band_df.to_csv(OUT / 'periodic_hydrogen_lowest_band.csv', index=False)
display(band_df)

if not bool(band_df['converged'].all()):
    bad = band_df.loc[~band_df['converged'], ['index','label','normalized_residual','iterations']]
    raise RuntimeError('Unconverged k-points:\n' + bad.to_string(index=False))

print(f"Band width over sampled path = {band_df['energy_Ry'].max()-band_df['energy_Ry'].min():.6f} Ry")
print(f"Maximum final residual       = {band_df['normalized_residual'].max():.3e}")

plot_band_structure(
    band_df, path,
    OUT / 'periodic_hydrogen_lowest_band.png',
    show=True,
)


# 8. Periodic electron density

Even when a Bloch wavefunction acquires a phase across a cell, its probability density is ordinarily periodic. The plot below tiles the Gamma-point density over a 3 x 3 set of cells.

The plotting theme intentionally matches the main paper: red-yellow density map, white contours, black proton markers, and the same figure dimensions and font hierarchy.


In [ ]:
plot_tiled_density(
    gamma.psi,
    A_BOHR,
    OUT / 'periodic_hydrogen_gamma_density_tiled.png',
    tiles=3,
    plane_axis='z',
    show=True,
)


# 9. Optional time-reversal validation

For a real scalar potential without a magnetic field, the spinless spectrum obeys E(k)=E(-k). This costs two additional relaxations and is enabled automatically in production mode.


In [ ]:
RUN_TIME_REVERSAL_CHECK = PRODUCTION

if RUN_TIME_REVERSAL_CHECK:
    ktest = np.array([0.31, 0.19, 0.11]) * pi_over_a
    plus = solve_ground_state(
        V_np, A_BOHR, ktest, gamma.psi,
        sigma=SIGMA, tolerance=TOL, max_iterations=MAX_ITER,
        check_every=CHECK_EVERY, verbose=False,
    )
    minus = solve_ground_state(
        V_np, A_BOHR, -ktest, np.conj(plus.psi.numpy()),
        sigma=SIGMA, tolerance=TOL, max_iterations=MAX_ITER,
        check_every=CHECK_EVERY, verbose=False,
    )
    delta = abs(plus.energy - minus.energy)
    print(f'E(+k) = {plus.energy:+.12f} Ry')
    print(f'E(-k) = {minus.energy:+.12f} Ry')
    print(f'|E(+k)-E(-k)| = {delta:.3e} Ry')
    assert plus.converged and minus.converged
    assert delta < 1e-8
else:
    print('Skipped in quick mode.')


# 10. Save reproducibility metadata

Nothing is written into the main paper results directories.


In [ ]:
metadata = {
    'case': 'bloch_periodic_one_electron_hydrogenic_lattice',
    'scope': 'single-particle boundary-condition demonstration; not many-electron metallic hydrogen',
    'n': N,
    'a_Bohr': A_BOHR,
    'h_Bohr': H,
    'sigma': SIGMA,
    'tolerance': TOL,
    'max_iterations': MAX_ITER,
    'points_per_segment': POINTS_PER_SEGMENT,
    'potential': V_meta,
    'gamma_energy_Ry': gamma.energy,
    'gamma_residual': gamma.residual,
    'gamma_iterations': gamma.iterations,
    'sampled_band_min_Ry': float(band_df['energy_Ry'].min()),
    'sampled_band_max_Ry': float(band_df['energy_Ry'].max()),
    'sampled_band_width_Ry': float(band_df['energy_Ry'].max()-band_df['energy_Ry'].min()),
    'max_band_residual': float(band_df['normalized_residual'].max()),
    'hermiticity_relative_error': herr,
    'max_free_electron_energy_error_Ry': float(free_test['abs_error_Ry'].max()),
    'max_free_electron_residual': float(free_test['normalized_residual'].max()),
    'tensorflow_version': tf.__version__,
}

with open(OUT / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, sort_keys=True)

print('Appendix outputs:')
for p in sorted(OUT.iterdir()):
    print(' -', p.name)


# Interpretation boundary

This appendix establishes only that:

1. the matrix-free finite-difference Hamiltonian remains Hermitian under Bloch-periodic wrapping;
2. the same normalized Rayleigh-residual relaxation converges to the lowest state of H_h(k);
3. the periodic hydrogenic model produces a reproducible lowest-band dispersion and periodic density.

No claim is made here about interacting metallic hydrogen, electron-electron correlation, structural phase stability, or a realistic many-electron Fermi surface.
